# DimUser

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import sys
import os

# Ensure the path points to 'src'
src_path = "/Workspace/Users/jaladikt006@gmail.com/spotifyAzureProject/my_project"

if src_path not in sys.path:
    sys.path.append(src_path)

# Option A: Import the class
from utils.transformations import reusable



##AUTO LOADER##

In [0]:
df_user1 = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimUser/checkpoint")\
    .load("abfss://bronze@vadlsgen2.dfs.core.windows.net/DimUser")

In [0]:
df_user1 = df_user1.withColumn("user_name", upper(col("user_name")))

In [0]:
df_user1_obj = reusable()

df_user1 = df_user1_obj.dropColumns(df_user1, ["_rescued_data"])
df_user1 = df_user1.drop_duplicates(["user_id"])


In [0]:
df_user1.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")

In [0]:
#display(df_user, checkpointLocation = "abfss://silver@vadlsgen2.dfs.core.windows.net/DimUser/data")

# DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimArtist/checkpoint")\
    .load("abfss://bronze@vadlsgen2.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art, ["_rescued_data"])
df_art = df_art.drop_duplicates(["artist_id"])

In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist")
    

In [0]:
# display(df_art, checkpointLocation = "abfss://silver@vadlsgen2.dfs.core.windows.net/DimArtist/checkpoint")

%md
# DimTrack

In [0]:
df_Track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimTrack/checkpoint")\
    .load("abfss://bronze@vadlsgen2.dfs.core.windows.net/DimTrack")

In [0]:
df_Track = df_Track.withColumn("durationFlag", when(col('duration_sec') > 150, 'low').when(col('duration_sec') < 150, 'medium').otherwise('high'))

df_Track = df_Track.withColumn("track_name",regexp_replace(col('track_name'),'-',''))

df_Track = reusable().dropColumns(df_Track, ["_rescued_data"])


In [0]:
spark.sql("DROP TABLE IF EXISTS spotify_cata.silver.df_track")

In [0]:
df_Track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")

# dim date

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimDate/checkpoint")\
    .load("abfss://bronze@vadlsgen2.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date, ["_rescued_data"])

df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@vadlsgen2.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")

# Fact Stream

In [0]:
df_FactStream = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/FactStream/checkpoint")\
    .load("abfss://bronze@vadlsgen2.dfs.core.windows.net/FactStream")

In [0]:
df_FactStream = reusable().dropColumns(df_FactStream, ["_rescued_data"])

df_FactStream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@vadlsgen2.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@vadlsgen2.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
SELECT *
FROM spotify_cata.gold.dimtrack -- ensure you run this query on a shared cluster or SQL warehouse that supports streaming tables
WHERE track_id IN (46, 5)